# Module 04: Data Structures Deep Dive — Solutions

Complete solutions to all exercises.

## Exercise 1: List vs Set Performance

In [ ]:
import random
import time

random.seed(42)

# Generate 100000 random integers
n = 100000
data_list = [random.randint(0, 1000000) for _ in range(n)]
data_set = set(data_list)

# Generate 100 test numbers
test_numbers = [random.randint(0, 1000000) for _ in range(100)]

# Time list membership
t0 = time.time()
list_results = []
for num in test_numbers:
    list_results.append(num in data_list)
t_list = time.time() - t0

# Time set membership
t0 = time.time()
set_results = []
for num in test_numbers:
    set_results.append(num in data_set)
t_set = time.time() - t0

print("List membership 100 checks:", round(t_list, 6), "sec")
print("Set membership 100 checks:", round(t_set, 6), "sec")
print("List is", round(t_list / t_set, 1), "x slower than set")
print("\nTotal items:", n)
print("Unique items:", len(data_set))
print("Results match:", list_results == set_results)

## Exercise 2: Class Distribution with Counter

In [ ]:
from collections import Counter

labels = [
    "cat", "dog", "bird", "cat", "dog", "cat", "fish", "bird",
    "cat", "dog", "cat", "cat", "dog", "bird", "fish", "cat",
    "dog", "dog", "cat", "bird", "cat", "dog", "cat", "fish",
    "cat", "dog", "cat", "cat", "dog", "dog", "bird", "cat",
    "rabbit", "cat", "dog", "cat", "hamster", "bird"
]

counts = Counter(labels)
total = len(labels)

print("=== Class Distribution ===")
print("Total samples:", total)
print()

# Top 3 most common
print("Top 3 classes:")
for cls, cnt in counts.most_common(3):
    pct = cnt / total * 100
    print("  " + cls + ":", cnt, "(" + str(round(pct, 1)) + "%)")

# Least common
least_common = counts.most_common()[-1]
print("\nLeast common class:", least_common[0], "(" + str(least_common[1]) + ")")

# Percentage for each
print("\nAll classes:")
for cls, cnt in counts.most_common():
    pct = cnt / total * 100
    bar = "#" * cnt
    print("  " + cls + ":", str(cnt).rjust(3), "(" + str(round(pct, 1)).rjust(4) + "%)", bar)

# Check imbalance
min_pct = min(counts.values()) / total * 100
if min_pct < 10:
    print("\nWARNING: Class imbalance detected!")
    print("  Least common class is only", round(min_pct, 1), "% of total")
else:
    print("\nNo severe class imbalance.")

## Exercise 3: Grouping with defaultdict

In [ ]:
from collections import defaultdict

data = [
    ("sales", 150, "2024-01-15 09:30:00"),
    ("sales", 200, "2024-01-15 10:15:00"),
    ("signups", 25, "2024-01-15 09:45:00"),
    ("sales", 175, "2024-01-15 09:55:00"),
    ("signups", 30, "2024-01-15 10:30:00"),
    ("pageviews", 500, "2024-01-15 09:00:00"),
    ("pageviews", 450, "2024-01-15 10:00:00"),
    ("signups", 20, "2024-01-15 11:00:00"),
    ("sales", 300, "2024-01-15 11:30:00"),
    ("pageviews", 600, "2024-01-15 11:15:00")
]

# 1. Group all values by category
by_category = defaultdict(list)
for cat, val, ts in data:
    by_category[cat].append(val)

print("=== Grouped by Category ===")
for cat, vals in sorted(by_category.items()):
    min_val = min(vals)
    max_val = max(vals)
    mean_val = sum(vals) / len(vals)
    print(cat + ": count=" + str(len(vals)) + " min=" + str(min_val) + " max=" + str(max_val) + " mean=" + str(round(mean_val, 2)))

# 2. Group by category AND hour
hierarchical = defaultdict(lambda: defaultdict(list))
for cat, val, ts in data:
    hour = ts.split(" ")[1].split(":")[0]
    hierarchical[cat][hour].append(val)

print("\n=== Grouped by Category + Hour ===")
for cat in sorted(hierarchical.keys()):
    print(cat + ":")
    for hour in sorted(hierarchical[cat].keys()):
        vals = hierarchical[cat][hour]
        print("  Hour", hour + ":", vals, "(mean:", round(sum(vals)/len(vals), 2), ")")

## Exercise 4: Sliding Window Statistics with deque

In [ ]:
from collections import deque
import random
import math

random.seed(42)

# Generate 1000 readings with injected spikes
base_readings = [random.random() * 10 + 20 for _ in range(1000)]  # baseline ~20-30

# Inject some anomalies (spikes) at known positions
anomaly_positions = [100, 300, 500, 700, 900]
for pos in anomaly_positions:
    base_readings[pos] = base_readings[pos] + 50  # big spike

window = deque(maxlen=50)
anomalies = []

for i, reading in enumerate(base_readings):
    window.append(reading)
    
    if len(window) == window.maxlen:
        win_mean = sum(window) / len(window)
        win_var = sum((x - win_mean) ** 2 for x in window) / len(window)
        win_std = math.sqrt(win_var) if win_var > 0 else 0
        
        # Anomaly: more than 3 std from window mean
        if abs(reading - win_mean) > 3 * win_std and win_std > 0:
            anomalies.append((i, round(reading, 2), round(win_mean, 2), round(win_std, 2)))

print("=== Anomaly Detection Results ===")
print("Total readings:", len(base_readings))
print("Window size:", 50)
print("Anomalies detected:", len(anomalies))
print()
if anomalies:
    print("Detected anomalies:")
    print("Index  | Value  | Window Mean | Window Std")
    print("-" * 50)
    for idx, val, mean, std in anomalies[:10]:
        print(str(idx).rjust(5), "|", str(val).rjust(6), "|", str(mean).rjust(11), "|", str(std).rjust(10))
    if len(anomalies) > 10:
        print("... and", len(anomalies) - 10, "more")

## Exercise 5: Top-K Feature Importance with heapq

In [ ]:
import heapq
import random

random.seed(42)

# Generate 50 features with random importance scores
feature_importance = {}
for i in range(50):
    feature_importance["feature_" + str(i)] = random.random()

# 1. Top 5 features
top_5 = heapq.nlargest(5, feature_importance, key=feature_importance.get)
print("=== Top 5 Features ===")
for feat in top_5:
    print("  " + feat + ":", round(feature_importance[feat], 4))

# 2. Bottom 3 features
bottom_3 = heapq.nsmallest(3, feature_importance, key=feature_importance.get)
print("\n=== Bottom 3 Features ===")
for feat in bottom_3:
    print("  " + feat + ":", round(feature_importance[feat], 4))

# 3. Features with importance > 0.5
high_importance = [feat for feat, score in feature_importance.items() if score > 0.5]
print("\n=== Features with importance > 0.5 ===")
print("  Count:", len(high_importance))
for feat in sorted(high_importance):
    print("  " + feat + ":", round(feature_importance[feat], 4))

# 4. Dict with only top-5
top_5_dict = {feat: feature_importance[feat] for feat in top_5}
print("\n=== Top 5 Dict ===")
print("  ", top_5_dict)

# 5. Cumulative importance of top-5
total_importance = sum(feature_importance.values())
top_5_sum = sum(feature_importance[feat] for feat in top_5)
print("\n=== Cumulative Importance ===")
print("  Total importance (all 50 features):", round(total_importance, 4))
print("  Top-5 sum:", round(top_5_sum, 4))
print("  Top-5 explains", round(top_5_sum / total_importance * 100, 1), "% of total importance")

## Exercise 6: Sorted List Operations with bisect

In [ ]:
import bisect
import random

random.seed(42)

# 1-2. Insert 20 random scores into sorted list
scores = []
for i in range(20):
    score = random.random()
    bisect.insort(scores, score)

print("=== Sorted Scores ===")
print("  ", [round(s, 4) for s in scores])

# 3. Find rank of a specific score
target = 0.5
rank = bisect.bisect_left(scores, target) + 1
print("\n=== Rank Finding ===")
print("  Score", target, "would rank at position", rank, "out of", len(scores))

# Find actual closest score
if rank <= len(scores):
    actual_rank_score = scores[rank - 1]
    print("  Score at rank", str(rank) + ":", round(actual_rank_score, 4))

# 4. Scores above 90th percentile
threshold_idx = int(len(scores) * 0.9)
threshold = scores[threshold_idx] if threshold_idx < len(scores) else 1.0
above_90th = scores[bisect.bisect_left(scores, threshold):]
print("\n=== Scores Above 90th Percentile ===")
print("  Threshold (90th %ile):", round(threshold, 4))
print("  Count:", len(above_90th))
print("  Scores:", [round(s, 4) for s in above_90th])

# 5. Scores in range [0.3, 0.7]
left = bisect.bisect_left(scores, 0.3)
right = bisect.bisect_right(scores, 0.7)
in_range = scores[left:right]
print("\n=== Scores in Range [0.3, 0.7] ===")
print("  Count:", len(in_range))
print("  Scores:", [round(s, 4) for s in in_range])
print("  Indices:", left, "to", right - 1)